# Visualizaciones de Evaluación
Matriz de Confusión · Curva ROC · Curva Precision-Recall

In [ ]:
!pip install -q seaborn

In [ ]:
# subir corpus_sentimiento_reviews.json desde tu computadora
from google.colab import files
uploaded = files.upload()

In [ ]:
import json, io
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
)

data = json.load(io.BytesIO(uploaded['corpus_sentimiento_reviews.json']))

reviews      = [item['texto']       for item in data['reviews']]
sentimientos = [item['sentimiento'] for item in data['reviews']]

In [ ]:
vectorizador = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
X = vectorizador.fit_transform(reviews)

X_train, X_test, y_train, y_test = train_test_split(
    X, sentimientos, test_size=0.3, random_state=42
)

modelo = LogisticRegression()
modelo.fit(X_train, y_train)

y_pred  = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, list(modelo.classes_).index('positivo')]
y_bin   = [1 if y == 'positivo' else 0 for y in y_test]

## Métricas desde cero (sin sklearn)

In [ ]:
def confusion_matrix_manual(y_true, y_pred, pos_label='positivo'):
    TP = sum(t == pos_label and p == pos_label for t, p in zip(y_true, y_pred))
    TN = sum(t != pos_label and p != pos_label for t, p in zip(y_true, y_pred))
    FP = sum(t != pos_label and p == pos_label for t, p in zip(y_true, y_pred))
    FN = sum(t == pos_label and p != pos_label for t, p in zip(y_true, y_pred))
    return TP, TN, FP, FN

def accuracy(y_true, y_pred):
    TP, TN, FP, FN = confusion_matrix_manual(y_true, y_pred)
    return (TP + TN) / (TP + TN + FP + FN)

def precision(y_true, y_pred, pos_label='positivo'):
    TP, _, FP, _ = confusion_matrix_manual(y_true, y_pred, pos_label)
    return TP / (TP + FP) if (TP + FP) > 0 else 0.0

def recall(y_true, y_pred, pos_label='positivo'):
    TP, _, _, FN = confusion_matrix_manual(y_true, y_pred, pos_label)
    return TP / (TP + FN) if (TP + FN) > 0 else 0.0

def f1(y_true, y_pred, pos_label='positivo'):
    p = precision(y_true, y_pred, pos_label)
    r = recall(y_true, y_pred, pos_label)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

In [ ]:
TP, TN, FP, FN = confusion_matrix_manual(y_test, y_pred)
print(f'Matriz de confusión: TP={TP}  TN={TN}  FP={FP}  FN={FN}')
print(f'Accuracy:  {accuracy(y_test, y_pred):.2f}')
print(f'Precision: {precision(y_test, y_pred):.2f}')
print(f'Recall:    {recall(y_test, y_pred):.2f}')
print(f'F1-score:  {f1(y_test, y_pred):.2f}')

## Matriz de Confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['positivo', 'negativo'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['positivo', 'negativo'],
            yticklabels=['positivo', 'negativo'])
plt.title('Matriz de Confusión')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

## Curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_bin, y_proba)
auc = roc_auc_score(y_bin, y_proba)
plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('Curva ROC')
plt.xlabel('FPR (falsos positivos)')
plt.ylabel('TPR / Recall')
plt.legend()
plt.show()

## Curva Precision-Recall

In [ ]:
pr_precision, pr_recall, _ = precision_recall_curve(y_bin, y_proba)
ap = average_precision_score(y_bin, y_proba)
plt.plot(pr_recall, pr_precision, label=f'AP = {ap:.2f}')
plt.title('Curva Precision-Recall')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.show()